# 🔍 Chapter 7: Unsupervised Learning
**Reference:** *Practical Statistics for Data Scientists (2nd Edition)* by Peter Bruce, Andrew Bruce, & Peter Gedeck

---

## 7.1 Introduction to Unsupervised Learning
In Chapters 4 and 5, we explored *Supervised Learning* (Regression and Classification), where the data has a known target variable ($Y$) that we want to predict. 

**Unsupervised Learning** deals with data where there is no target variable. The algorithm is simply given a set of features ($X$) and is asked to find hidden structures, natural groupings, or condensed representations of the data.

Unsupervised learning is primarily used for:
1. **Dimensionality Reduction:** Squashing many features down to a few meaningful components (PCA).
2. **Clustering:** Grouping rows of data together based on their similarity (Customer Segmentation, Anomaly Detection).

In this chapter, we will explore the major clustering algorithms and the critical importance of feature scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from scipy.cluster.hierarchy import dendrogram, linkage

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
np.random.seed(42)

print("Libraries successfully imported for Chapter 7.")

## 7.2 Principal Components Analysis (PCA)
While we covered the mathematical roots of PCA in Chapters 13 and 15, the book emphasizes its use in Unsupervised Learning as an exploratory tool and a preprocessing step.

PCA takes a dataset with $N$ features and creates $N$ new features called **Principal Components (PCs)**. 
- PC1 captures the maximum possible variance in the data.
- PC2 is strictly orthogonal (perpendicular) to PC1 and captures the maximum remaining variance.

By keeping only the first few PCs, we can visualize highly complex datasets in 2D or 3D, and remove correlated noise.

In [ ]:
# Generate a highly correlated 3D dataset
mean = [0, 0, 0]
cov = [[1, 0.9, 0.8], [0.9, 1, 0.7], [0.8, 0.7, 1]]
data_3d = np.random.multivariate_normal(mean, cov, 300)

# Apply PCA to reduce from 3D to 2D
pca = PCA(n_components=2)
data_2d = pca.fit_transform(data_3d)

print("Explained Variance Ratio by Component:", np.round(pca.explained_variance_ratio_, 3))
print(f"Total Variance Retained: {np.sum(pca.explained_variance_ratio_)*100:.1f}%")

plt.figure(figsize=(7, 5))
plt.scatter(data_2d[:, 0], data_2d[:, 1], alpha=0.6, color='teal')
plt.title('PCA: 3D Data Compressed to 2D')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

## 7.3 K-Means Clustering
K-Means is the most popular clustering algorithm. You specify the number of clusters ($K$) upfront, and the algorithm groups the data by minimizing the distance between the data points and their cluster centers (centroids).

**The Algorithm:**
1. Randomly drop $K$ centroids into the data space.
2. Assign every data point to the centroid it is closest to.
3. Move the centroid to the exact mean (average) location of all points assigned to it.
4. Repeat steps 2 and 3 until the centroids stop moving.

**Choosing K (The Elbow Method):**
To find the optimal number of clusters, we plot the Sum of Squared Errors (Inertia) for different values of $K$. We look for the "elbow" where adding more clusters yields diminishing returns.

In [ ]:
# Create synthetic data with 4 hidden clusters
X_blobs, y_true = make_blobs(n_samples=500, centers=4, cluster_std=1.2, random_state=42)

# 1. The Elbow Method to find optimal K
inertia = []
k_values = range(1, 8)
for k in k_values:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(X_blobs)
    inertia.append(kmeans_temp.inertia_)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(k_values, inertia, 'o-k')
plt.title('The Elbow Method for K-Means')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Sum of Squared Distances)')
plt.axvline(4, color='red', linestyle='--') # The elbow is at K=4

# 2. Apply K-Means with K=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_blobs)

plt.subplot(1, 2, 2)
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=cluster_labels, cmap='viridis', alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            s=200, c='red', marker='X', label='Centroids')
plt.title('K-Means Clustering Result (K=4)')
plt.legend()

plt.tight_layout()
plt.show()

## 7.4 Hierarchical Clustering
A major limitation of K-Means is that you must guess $K$ upfront. **Hierarchical Clustering** does not require $K$. Instead, it creates a tree of relationships called a **Dendrogram**.

**Agglomerative (Bottom-Up) Algorithm:**
1. Start by treating every single data point as its own cluster (e.g., 500 points = 500 clusters).
2. Find the two clusters that are closest together and merge them.
3. Repeat step 2 until all points are merged into a single, giant cluster.

By slicing the dendrogram horizontally at a specific height, you can choose exactly how many clusters you want after visualizing the entire tree structure.

In [ ]:
# To make the dendrogram readable, we will use a small subset of the data (40 points)
X_small = X_blobs[:40]

# Perform hierarchical/agglomerative clustering using 'Ward' linkage
# Ward minimizes the variance of the clusters being merged
Z = linkage(X_small, method='ward')

plt.figure(figsize=(10, 5))
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Data Point Index')
plt.ylabel('Distance (Height)')

# Draw the dendrogram
dendrogram(Z, leaf_rotation=90, leaf_font_size=10)

# Draw a cut-off line showing where we would split to get 3 clusters
plt.axhline(y=10, color='red', linestyle='--', label='Cut to yield 3 clusters')
plt.legend()
plt.show()

print("Interpretation: Moving up the y-axis represents merging clusters at larger distances. The longer the vertical lines, the more distinct the clusters are.")

## 7.5 Model-Based Clustering (Gaussian Mixture Models)
K-Means has a hidden mathematical bias: it measures distance in perfect circles (Euclidean distance). If your clusters are elliptical (oval-shaped) or overlapping, K-Means will fail dramatically.

**Gaussian Mixture Models (GMM)** solve this by assuming the data is generated by a mix of several Multivariate Normal Distributions. 
- Instead of assigning a point to a rigid cluster (Hard Clustering), GMM calculates the *probability* that a point belongs to a cluster (Soft Clustering).
- GMM can stretch and rotate its clusters to fit ovals, lines, and complex distributions.

In [ ]:
# Create an elliptical (stretched) dataset where K-Means would normally fail
np.random.seed(4)
X_stretched = np.dot(np.random.randn(300, 2), [[0.6, -0.6], [-0.4, 0.8]]) 
X_shifted = np.random.randn(300, 2) + np.array([4, 4])
X_gmm = np.vstack([X_stretched, X_shifted])

# Fit Gaussian Mixture Model
gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
gmm_labels = gmm.fit_predict(X_gmm)

# Compare with K-Means
kmeans_gmm = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_gmm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot K-Means failure
axes[0].scatter(X_gmm[:, 0], X_gmm[:, 1], c=kmeans_gmm, cmap='coolwarm', alpha=0.6)
axes[0].set_title("K-Means (Fails on Elliptical Data)")

# Plot GMM success
axes[1].scatter(X_gmm[:, 0], X_gmm[:, 1], c=gmm_labels, cmap='coolwarm', alpha=0.6)
axes[1].set_title("Gaussian Mixture Model (Adapts to Covariance)")

plt.show()
print("Notice how K-Means forces a harsh, straight line to separate the data, slicing the stretched cluster in half. GMM correctly identifies the overlapping distribution.")

## 7.6 Scaling and Categorical Variables

**1. The Scaling Problem:**
Clustering algorithms are almost entirely driven by distance calculations (Euclidean space). If you have two variables, `Age` (0-100) and `Income` (0-1,000,000), the algorithm will cluster people *exclusively* based on Income, treating Age as practically zero.

You **must** scale your data (e.g., using `StandardScaler` to calculate Z-scores) so that every feature has a mean of 0 and a standard deviation of 1 before clustering.

**2. Categorical Variables:**
Standard Euclidean distance does not work on categories (e.g., Distance between 'Red' and 'Blue').
- Do NOT just convert them to numbers (1=Red, 2=Blue, 3=Green), because it implies math (Red < Blue).
- Do use One-Hot Encoding (binary 0/1 columns).
- Better yet, use **Gower's Distance**, a specialized mathematical formula that handles mixed data types (numerical and categorical) natively, which can then be fed into a Hierarchical Clustering algorithm.

In [ ]:
# Demonstrating the fatal flaw of unscaled data
unscaled_data = np.array([
    [25, 50000],   # Young, Low Income
    [26, 150000],  # Young, High Income
    [75, 50000]    # Old, Low Income
])

# Let's calculate the Euclidean distance between Point 1 and Point 2
dist_1_2 = np.linalg.norm(unscaled_data[0] - unscaled_data[1])
# Let's calculate the Euclidean distance between Point 1 and Point 3
dist_1_3 = np.linalg.norm(unscaled_data[0] - unscaled_data[2])

print(f"Distance between (Age 25, 50k) and (Age 26, 150k): {dist_1_2:,.2f}")
print(f"Distance between (Age 25, 50k) and (Age 75, 50k) : {dist_1_3:,.2f}")

print("\nThe algorithm thinks a 75-year-old is IDENTICAL to a 25-year-old, because the massive income difference dominates the math. ALWAYS scale your data for clustering!")

### Conclusion of Chapter 7
Unsupervised learning is the realm of data exploration and pattern discovery. 

**Key Takeaways:**
1. **PCA** transforms raw features into orthogonal components based on variance, enabling visualization and dimensionality reduction.
2. **K-Means** is fast and scalable, but struggles with non-spherical clusters and requires you to guess $K$.
3. **Hierarchical Clustering** yields beautiful, interpretable dendrograms that let you see the clustering hierarchy, but is too slow for massive datasets.
4. **Gaussian Mixture Models (GMM)** use probability and covariance to gracefully handle overlapping, complex clusters.
5. **Data Preparation is Everything:** If you do not standardize numeric variables and correctly encode categorical variables, your clustering model will output mathematical garbage.